# `web.search` Tool: Real + Mocked Tests

Covers task backlog §3.3. Two layers, deliberately different from §3.2's approach:

1. **One real call** through the real MCP server (same subprocess/stdio pattern as
   §3.1/§3.2), to prove the actual Tavily integration works end-to-end -- kept to exactly
   one call, since Tavily's free tier is a real, finite resource (1,000 credits/month),
   same cost-consciousness principle applied to OpenAI calls in §2.2
2. **Mocked unit tests** for caching and retry/backoff -- the backlog's own wording for
   §3.3 explicitly asks for "mocked HTTP responses" (unlike §3.2's "real index instead of
   a fixture" substitution). Mocking is the right call *here* specifically: you can't
   reliably force a real rate-limit response on demand, and testing backoff timing
   against the real API would be slow and non-deterministic

In [1]:
# !pip install mcp tavily-python

## Real call through the real MCP server (1 Tavily credit spent)

In [2]:
import json
import os
import sys

from mcp.client.session import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

SERVER_PATH = os.path.join(os.getcwd(), "mcp_server.py")
server_params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.call_tool("web.search", {"query": "Wildkin kids insulated lunch box price", "top_k": 3})

assert not result.is_error, f"unexpected tool error: {result.content[0].text}"
payload = json.loads(result.content[0].text)
print(f"{len(payload['results'])} results, cache_hit={payload['cache_hit']}")
for r in payload["results"]:
    print(f"  {r['title'][:55]!r:57} price={r['price']} availability={r['availability']}")
    print(f"    {r['url']}")

assert len(payload["results"]) > 0
assert all(any(d in r["url"] for d in ["amazon.com", "walmart.com", "target.com", "ebay.com", "bestbuy.com"]) for r in payload["results"]), "domain allowlist should hold"
print("\n[OK] real Tavily call through the real MCP server, domain allowlist respected.")

3 results, cache_hit=False
  'Wildkin Kids Insulated Lunch Box – Durable and Reusable' price=None availability=None
    https://www.target.com/p/wildkin-lunch-box-for-kids/-/A-86483178
  'Wildkin Kids Insulated Lunch Box Bag (Grey Tweed) : Tar' price=None availability=None
    https://www.target.com/p/wildkin-kids-insulated-lunch-box-bag-grey-tweed/-/A-83818052
  'Wildkin Kids Insulated Lunch Box Bag (Magical Unicorns)' price=None availability=None
    https://www.target.com/p/wildkin-kids-insulated-lunch-box-bag-magical-unicorns/-/A-83818038

[OK] real Tavily call through the real MCP server, domain allowlist respected.


## Mocked tests: caching, backoff, domain allowlist wiring, price/availability heuristics

In-process (not through the MCP subprocess -- mocking the Tavily client directly is
simpler and doesn't need a second server round-trip). 0 real Tavily calls in this section.

In [3]:
from unittest.mock import MagicMock, patch

import web_search_tool as wst

In [4]:
# Caching: second call within TTL should be served from cache, not hit the client again
wst._cache.clear()
fake_client = MagicMock()
fake_client.search.return_value = {"results": [{"title": "T", "url": "https://amazon.com/x", "content": "$9.99, in stock"}]}

with patch.object(wst, "_get_client", return_value=fake_client):
    r1 = wst.web_search_impl("unique cache test query", top_k=2)
    r2 = wst.web_search_impl("unique cache test query", top_k=2)

assert fake_client.search.call_count == 1, "second call should have been served from cache, not re-called Tavily"
assert r1["cache_hit"] is False and r2["cache_hit"] is True
print("[OK] caching: 1 real client call made, 2nd request served from cache")

[OK] caching: 1 real client call made, 2nd request served from cache


In [5]:
# Backoff: retries on a transient timeout, succeeds on the 2nd attempt
wst._cache.clear()
flaky_client = MagicMock()
flaky_client.search.side_effect = [wst.TavilyTimeoutError(1.0), {"results": []}]

with patch.object(wst, "_get_client", return_value=flaky_client), patch.object(wst.time, "sleep") as mock_sleep:
    wst.web_search_impl("unique backoff success query", top_k=1)

assert flaky_client.search.call_count == 2, "should have retried once after the transient timeout"
assert mock_sleep.called, "should have backed off before retrying"
print("[OK] backoff: retried after a transient timeout, succeeded on the 2nd attempt")

# Persistent failure: exhausts MAX_RETRIES and raises a typed ToolError (\u00a73.4's error
# contract), not the raw Tavily exception -- doesn't retry forever or swallow the error
wst._cache.clear()
always_fails = MagicMock()
always_fails.search.side_effect = wst.TavilyTimeoutError(1.0)

with patch.object(wst, "_get_client", return_value=always_fails), patch.object(wst.time, "sleep"):
    try:
        wst.web_search_impl("unique persistent failure query", top_k=1)
        raised = False
    except wst.ToolError as e:
        raised = True
        error_type = e.error_type

assert raised, "should raise after exhausting retries, not silently fail"
assert error_type == "timeout", "should surface as a typed ToolError with error_type='timeout' (\u00a73.4)"
assert always_fails.search.call_count == wst.MAX_RETRIES
print(f"[OK] backoff: exhausted all {wst.MAX_RETRIES} retries, raised a typed ToolError(error_type='timeout'), not swallowed")

[OK] backoff: retried after a transient timeout, succeeded on the 2nd attempt
[OK] backoff: exhausted all 3 retries, raised a typed ToolError(error_type='timeout'), not swallowed


In [6]:
# Domain allowlist is actually passed to Tavily, not just defined and unused
wst._cache.clear()
capture_client = MagicMock()
capture_client.search.return_value = {"results": []}

with patch.object(wst, "_get_client", return_value=capture_client):
    wst.web_search_impl("unique domain allowlist query", top_k=1)

call_kwargs = capture_client.search.call_args.kwargs
assert call_kwargs["include_domains"] == wst.ALLOWED_DOMAINS
print("[OK] ALLOWED_DOMAINS is actually forwarded as Tavily's include_domains, not dead code")

[OK] ALLOWED_DOMAINS is actually forwarded as Tavily's include_domains, not dead code


In [7]:
# Price/availability heuristics -- pure functions, no mocking needed. Test strings are
# shaped like the real snippet seen in this notebook's own live call above
# ("The highest price is $35 $ - $ Category Lunch Boxes ... Sold out").
assert wst._extract_price("The highest price is $35 $ - $ Category Lunch Boxes") == 35.0
assert wst._extract_price("no dollar amount in this text") is None
assert wst._extract_availability("Standard Lunch $29.99 Sold out") == "out_of_stock"
assert wst._extract_availability("In stock now, ships today") == "in_stock"
assert wst._extract_availability("just a generic sentence") is None
print("[OK] price/availability heuristics behave correctly on real-shaped snippet text")
print("\nAll §3.3 tests passed: 1 real Tavily call + 6 mocked/pure cases, 0 additional real API cost.")

[OK] price/availability heuristics behave correctly on real-shaped snippet text

All §3.3 tests passed: 1 real Tavily call + 6 mocked/pure cases, 0 additional real API cost.


## Next steps

- §3.4: structured logging, secret scrubbing, typed error contract, MCP README
- The Retriever swap (§2.4's open item) + persistent server/session lifecycle -- planned
  as one consolidated piece of work covering both tools + conflict reconciliation together,
  not done as part of §3.3